In [ ]:
import pandas as pd
import glob
import itertools
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches



In [ ]:
!pwd

In [ ]:
result_csv = glob.glob('results/result*.csv')
result_csv

In [ ]:
df_vae = pd.read_csv('results/result_nn_vae.csv')
df_vae

In [ ]:
df_selec = pd.read_csv('results/result_nn_r6.csv')
df_selec

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.lines as mlines

df_vae['Type'] = 'VAE'
df_selec['Type'] = 'Select'

combined_df = pd.concat([df_selec, df_vae])

# Setting font and plot sizes as per the features extracted from the previous code
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 30  # Significantly increased font size
plt.rcParams["axes.labelsize"] = 30  # Increased label size
plt.rcParams["xtick.labelsize"] = 30  # Increased XTick size
plt.rcParams["ytick.labelsize"] = 30  # Increased YTick size

# Adjust figure size and DPI for better layout
plt.figure(figsize=(24, 12), dpi=300)  # Adjust figure size here

# Create a larger and fancier violin plot
sns.violinplot(x='Type', y='average_probability', data=combined_df, palette={'VAE': 'blue', 'Select': 'green'})
plt.ylabel('Average probability')  # Changing y-axis label
plt.xlabel('')  # Removing x-axis label

# Enhancing the border of the plot
ax = plt.gca()
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(4)  # Adjusted linewidth
    spine.set_edgecolor('black')


num_samples_vae = len(df_vae)
num_samples_selec = len(df_selec)

# # Create legend handles
# legend_handles = [

#     mpatches.Patch(color='green', label=f'Select: {num_samples_selec}'),
#         mpatches.Patch(color='blue', label=f'VAE: {num_samples_vae}')
# ]

# # Add the legend to the plot
# plt.legend(handles=legend_handles, title='Number of Samples', fontsize=30, title_fontsize=30,
#            loc='lower center', ncol=7, bbox_to_anchor=(0.5, -0.25),frameon=False)

y_max = combined_df['average_probability'].max()
text_offset = y_max * 0.05  # Offset to position text above the highest point in the plot

# Place the text for each violin plot
ax.text(0, y_max + text_offset-0.047, f'{num_samples_selec}', ha='right', va='center', color='green', fontsize=30)

ax.text(1, y_max + text_offset+0.047, f'{num_samples_vae}', ha='right', va='center', color='blue', fontsize=30)
    
plt.tight_layout(pad=4.0)

plt.show()


In [ ]:
df_seq_r6 = pd.read_csv('result_r6_80.csv')['sequence']
df_seq_vae = pd.read_csv('result_vae_send.csv')['sequence']
df_sequences  = pd.concat([df_seq_r6,df_seq_vae])
df_sequences= pd.DataFrame(df_sequences, columns=['sequence'])

df_sequences['SeqID'] = [f"S{i}" for i in range(1,len(df_sequences)+1)]
df_sequences.reset_index(drop=True, inplace=True)
df_sequences

In [ ]:
merged_df = pd.merge(df_sequences, combined_df, on='sequence', how='inner')
merged_df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Setting font and plot sizes
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 30
plt.rcParams["axes.labelsize"] = 30
plt.rcParams["xtick.labelsize"] = 30
plt.rcParams["ytick.labelsize"] = 30

# Extracting h5 columns for boxplot
h5_columns = [col for col in merged_df.columns if col.endswith('.h5')]

# Melting the DataFrame for easier plotting
melted_df = merged_df.melt(id_vars=["SeqID", "Type"], value_vars=h5_columns, 
                    var_name="Model", value_name="Value")

# Creating the plot with adjusted settings
plt.figure(figsize=(24, 12), dpi=300)
ax = sns.boxplot(x="SeqID", y="Value", hue="Type", data=melted_df, 
                 palette={"VAE": "blue", "Select": "green"})

# Setting alpha for boxplot patches
for patch in ax.artists:
    patch.set_alpha(0.1)

# Adding individual points
sns.stripplot(x="SeqID", y="Value", hue="Type", data=melted_df, 
              palette={"VAE": "lightblue", "Select": "lightgreen"}, dodge=True, jitter=True, alpha=1)

plt.xlabel("Sequence ID")
plt.ylabel("Probability")

# Adjusting plot border
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(4)
    spine.set_edgecolor('black')

# Adjusting legend
handles, labels = ax.get_legend_handles_labels()
plt.legend(handles[:2], labels[:2], loc='upper left', fontsize=30, title_fontsize=30, ncols=2, bbox_to_anchor=(0, 1.1), frameon=False)

# Adjusting the color of the fliers (outliers)
for artist in ax.lines:
    artist.set_color('black')
    artist.set_alpha(0.7)

plt.tight_layout()
plt.show()


In [ ]:
unique_seq_ids = sorted(merged_df['SeqID'].unique())
print("Unique Sequence IDs:", unique_seq_ids)

# Check if there is a jump or missing value between S8 and S9
if 'S9' not in unique_seq_ids:
    print("The 'SeqID' S9 is missing from the dataset.")

In [ ]:
filtered_df_box = combined_df[combined_df['sequence'].isin(df_sequences.tolist())]
filtered_df_box

In [ ]:
def are_identical(df1, df2):
    return df1.equals(df2)

# Load all CSV files into DataFrames
dataframes = {filename: pd.read_csv(filename) for filename in result_csv}

# Comparing each pair of CSV files
identical_files = []
for (file1, df1), (file2, df2) in itertools.combinations(dataframes.items(), 2):
    if are_identical(df1, df2):
        identical_files.append((file1, file2))

# Print out the results
for pair in identical_files:
    print(f"The files {pair[0]} and {pair[1]} are identical.")